# 03 - Gold assembly

Membentuk tabel per jam, fitur, target, dan dataset version.

In [ ]:
window_start = "2015-10-01"
window_end = "2015-10-31"

In [ ]:
# Parameters
window_start = "2015-10-01"
window_end = "2015-10-31"

# Configuration
from datetime import datetime, timedelta, timezone
import math
from functools import reduce

from pyspark.sql import Window
from pyspark.sql import functions as F

STORAGE_ACCOUNT = "sthidrogkiq7v7oc3lns"
SILVER = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net"
GOLD = f"abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net"
TIMEZONE = "America/Los_Angeles"
FEATURE_LOOKBACK_HOURS = 168
RUN_ID = datetime.now(timezone.utc).strftime("gold-%Y%m%dT%H%M%SZ")

spark.conf.set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
spark.conf.set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

STATIONS = [
    ("11525670", "Indian C near Douglas City, CA", 40.65178087, -122.9144343),
    ("11150500", "Salinas River near Bradley, CA", 35.9302434, -120.8687915),
    ("11454000", "Putah Creek near Winters, CA", 38.5151562, -122.0818793),
]

METEO_COLUMNS = [
    "precip_mm", "rain_mm", "temp_c", "rh_pct", "dewpoint_c",
    "soil_moist_0_7", "soil_moist_7_28", "soil_moist_28_100",
    "et0_mm", "wind_kmh", "cloud_pct", "swrad_wm2", "pressure_hpa",
]

STAGE_LAGS = (1, 2, 3, 6, 12, 24, 48, 72, 168)
STAGE_WINDOWS = (3, 6, 12, 24, 72, 168)
STAGE_DIFFS = (1, 3, 6, 24)
DISCHARGE_LAGS = (1, 3, 6, 24, 72)
PRECIP_WINDOWS = (1, 3, 6, 24, 72, 168)
MIN_PERIOD_RATIO = 0.9
STAGE_COLUMN = "stage_mean_ft"
PRECIP_COLUMN = "precip_mm"
DISCHARGE_COLUMN = "discharge_model_cfs"

FEATURE_COLS = [
    "stage_now",
    "stage_lag_1", "stage_lag_2", "stage_lag_3", "stage_lag_6",
    "stage_lag_12", "stage_lag_24", "stage_lag_48", "stage_lag_72", "stage_lag_168",
    "stage_mean_3", "stage_mean_6", "stage_mean_12", "stage_mean_24", "stage_mean_72", "stage_mean_168",
    "stage_max_3", "stage_max_6", "stage_max_12", "stage_max_24", "stage_max_72", "stage_max_168",
    "stage_diff_1", "stage_diff_3", "stage_diff_6", "stage_diff_24",
    "discharge_lag_1", "discharge_lag_3", "discharge_lag_6", "discharge_lag_24", "discharge_lag_72",
    "precip_sum_1", "precip_sum_3", "precip_sum_6", "precip_sum_24", "precip_sum_72", "precip_sum_168",
    "precip_sum_24_lag_24", "precip_sum_72_lag_24",
    "soil_moist_0_7", "soil_moist_7_28", "temp_c", "et0_mm", "rh_pct",
    "hour_sin", "hour_cos", "doy_sin", "doy_cos",
]
TARGET_COLS = ["y_h1", "y_h6", "y_h24"]

start_ts = datetime.fromisoformat(window_start + "T00:00:00+00:00")
end_ts = datetime.fromisoformat(window_end + "T23:00:00+00:00")
lookback_ts = start_ts - timedelta(hours=FEATURE_LOOKBACK_HOURS)
start_utc = start_ts.strftime("%Y-%m-%d %H:%M:%S")
end_utc = end_ts.strftime("%Y-%m-%d %H:%M:%S")
lookback_utc = lookback_ts.strftime("%Y-%m-%d %H:%M:%S")
start_date = window_start
end_date = window_end


def read_delta(path):
    return spark.read.format("delta").load(path)


stage = (
    read_delta(f"{SILVER}/delta/usgs/continuous/parameter_code=00065")
    .filter((F.col("time_utc") >= lookback_utc) & (F.col("time_utc") <= end_utc))
)
discharge = (
    read_delta(f"{SILVER}/delta/usgs/continuous/parameter_code=00060")
    .filter((F.col("time_utc") >= lookback_utc) & (F.col("time_utc") <= end_utc))
)
daily = (
    read_delta(f"{SILVER}/delta/usgs/daily/parameter_code=00060")
    .filter((F.col("date_local") >= start_date) & (F.col("date_local") <= end_date))
    .dropDuplicates(["station_id", "date_local"])
)
meteo = (
    read_delta(f"{SILVER}/delta/open-meteo")
    .filter((F.col("time_utc") >= lookback_utc) & (F.col("time_utc") <= end_utc))
    .select(
        "station_id", "time_utc",
        F.col("precipitation").alias("precip_mm"),
        F.col("rain").alias("rain_mm"),
        F.col("temperature_2m").alias("temp_c"),
        F.col("relative_humidity_2m").alias("rh_pct"),
        F.lit(None).cast("double").alias("dewpoint_c"),
        F.col("soil_moisture_0_to_7cm").alias("soil_moist_0_7"),
        F.col("soil_moisture_7_to_28cm").alias("soil_moist_7_28"),
        F.col("soil_moisture_28_to_100cm").alias("soil_moist_28_100"),
        F.col("et0_fao_evapotranspiration").alias("et0_mm"),
        F.col("wind_speed_10m").alias("wind_kmh"),
        F.col("cloud_cover").alias("cloud_pct"),
        F.col("shortwave_radiation").alias("swrad_wm2"),
        F.col("surface_pressure").alias("pressure_hpa"),
    )
    .dropDuplicates(["station_id", "time_utc"])
)

stations_df = spark.createDataFrame(
    [(site, name, lat, lon) for site, name, lat, lon in STATIONS],
    "station_id string, station_name string, latitude double, longitude double",
)
hours = spark.sql(
    f"select explode(sequence(timestamp'{lookback_utc}', timestamp'{end_utc}', interval 1 hour)) as time_utc"
)
skeleton = (
    stations_df.select("station_id", "station_name", "latitude", "longitude")
    .crossJoin(hours)
    .withColumn("time_local", F.from_utc_timestamp("time_utc", TIMEZONE))
    .withColumn("date_local", F.date_format("time_local", "yyyy-MM-dd"))
    .withColumn("date_utc", F.date_format("time_utc", "yyyy-MM-dd"))
)

stage_hourly = (
    stage.withColumn("time_utc", F.date_trunc("hour", "time_utc"))
    .groupBy("station_id", "time_utc")
    .agg(
        F.avg("value").alias("stage_mean_ft"),
        F.max("value").alias("stage_max_ft"),
        F.min("value").alias("stage_min_ft"),
        F.count("value").cast("long").alias("stage_n"),
    )
)
discharge_hourly = (
    discharge.withColumn("time_utc", F.date_trunc("hour", "time_utc"))
    .groupBy("station_id", "time_utc")
    .agg(
        F.avg("value").alias("discharge_mean_cfs"),
        F.count("value").cast("long").alias("discharge_n"),
    )
)
daily_hourly = (
    skeleton.select("station_id", "time_utc", "date_local").distinct()
    .join(daily.select("station_id", "date_local", "value"), ["station_id", "date_local"], "left")
    .select("station_id", "time_utc", F.col("value").alias("discharge_daily_cfs"))
)

frame = (
    skeleton
    .join(stage_hourly, ["station_id", "time_utc"], "left")
    .join(discharge_hourly, ["station_id", "time_utc"], "left")
    .join(daily_hourly, ["station_id", "time_utc"], "left")
    .join(meteo, ["station_id", "time_utc"], "left")
    .withColumn("discharge_model_cfs", F.coalesce("discharge_mean_cfs", "discharge_daily_cfs"))
    .withColumn("discharge_grain", F.when(F.col("discharge_mean_cfs").isNotNull(), "hourly").otherwise("daily"))
)


def ordering():
    return Window.partitionBy("station_id").orderBy("time_utc")


def rolling(n):
    return ordering().rowsBetween(-(n - 1), 0)


def min_periods(n):
    return max(1, math.ceil(MIN_PERIOD_RATIO * n))


def rolling_value(column, n, how):
    window = rolling(n)
    aggregate = {"mean": F.avg, "max": F.max, "sum": F.sum}[how](F.col(column)).over(window)
    samples = F.count(F.col(column)).over(window)
    return F.when(samples >= min_periods(n), aggregate).otherwise(F.lit(None)).cast("double")


def add_features(input_frame):
    stage_col = F.col(STAGE_COLUMN)
    discharge_col = F.col(DISCHARGE_COLUMN)
    precip_col = F.col(PRECIP_COLUMN)
    features = [stage_col.cast("double").alias("stage_now")]
    for lag in STAGE_LAGS:
        features.append(F.lag(stage_col, lag).over(ordering()).cast("double").alias(f"stage_lag_{lag}"))
    for window in STAGE_WINDOWS:
        features.append(rolling_value(STAGE_COLUMN, window, "mean").alias(f"stage_mean_{window}"))
        features.append(rolling_value(STAGE_COLUMN, window, "max").alias(f"stage_max_{window}"))
    for diff in STAGE_DIFFS:
        features.append((stage_col - F.lag(stage_col, diff).over(ordering())).cast("double").alias(f"stage_diff_{diff}"))
    for lag in DISCHARGE_LAGS:
        features.append(F.lag(discharge_col, lag).over(ordering()).cast("double").alias(f"discharge_lag_{lag}"))
    for window in PRECIP_WINDOWS:
        features.append(rolling_value(PRECIP_COLUMN, window, "sum").alias(f"precip_sum_{window}"))
    hour = F.hour(F.col("time_local"))
    day_of_year = F.dayofyear(F.col("time_local"))
    features.extend([
        F.sin(2 * math.pi * hour / 24).cast("double").alias("hour_sin"),
        F.cos(2 * math.pi * hour / 24).cast("double").alias("hour_cos"),
        F.sin(2 * math.pi * day_of_year / 365.25).cast("double").alias("doy_sin"),
        F.cos(2 * math.pi * day_of_year / 365.25).cast("double").alias("doy_cos"),
    ])
    output = input_frame.select("*", *features)
    output = output.withColumn("_precip_24", rolling_value(PRECIP_COLUMN, 24, "sum"))
    output = output.withColumn("_precip_72", rolling_value(PRECIP_COLUMN, 72, "sum"))
    output = output.withColumn(
        "precip_sum_24_lag_24",
        F.lag("_precip_24", 24).over(ordering()).cast("double"),
    )
    output = output.withColumn(
        "precip_sum_72_lag_24",
        F.lag("_precip_72", 24).over(ordering()).cast("double"),
    )
    output = output.drop("_precip_24", "_precip_72")
    for horizon in (1, 6, 24):
        output = output.withColumn(f"y_h{horizon}", F.lead(stage_col, horizon).over(ordering()).cast("double"))
    return output


features = add_features(frame)
non_null = reduce(lambda left, right: left & right, [F.col(column).isNotNull() for column in FEATURE_COLS + TARGET_COLS])
features = (
    features
    .withColumn("is_trainable", F.when(non_null, True).otherwise(False))
    .withColumn("is_partial", F.when(F.col("stage_n") < 3, True).otherwise(False))
    .withColumn("dataset_version", F.concat(F.lit("g1-"), F.md5(F.concat_ws("|", F.lit(window_start), F.lit(window_end))).substr(1, 8)))
)

version = features.select("dataset_version").first()["dataset_version"]
hourly_columns = [
    "station_id", "station_name", "latitude", "longitude", "time_utc", "time_local",
    "date_local", "date_utc", "stage_mean_ft", "stage_max_ft", "stage_min_ft", "stage_n",
    "discharge_mean_cfs", "discharge_daily_cfs", "discharge_model_cfs", "discharge_grain",
    *METEO_COLUMNS,
]
station_hourly = features.select(*hourly_columns)

hourly_path = f"{GOLD}/delta/station_hourly"
features_path = f"{GOLD}/delta/features"
def write_window(frame, path):
    predicate = f"date_utc >= '{window_start}' AND date_utc <= '{window_end}'"
    writer = (
        frame.write.format("delta").mode("overwrite")
        .partitionBy("station_id", "date_utc")
    )
    try:
        from delta.tables import DeltaTable
        if DeltaTable.isDeltaTable(spark, path):
            writer = writer.option("replaceWhere", predicate)
    except Exception:
        pass
    writer.save(path)

write_window(station_hourly, hourly_path)
write_window(features, features_path)

trainable = features.filter("is_trainable").count()
total = features.count()
version_row = [(
    RUN_ID, version, window_start, window_end, total, trainable,
    datetime.now(timezone.utc).isoformat(),
)]
spark.createDataFrame(
    version_row,
    "run_id string, dataset_version string, start_date string, end_date string, rows long, rows_trainable long, created_at string",
).write.format("delta").mode("append").save(f"{GOLD}/delta/dataset_versions")

dq_rows = [
    ("features", "row_count", total, RUN_ID),
    ("features", "trainable_rows", trainable, RUN_ID),
    ("features", "soft_fail_not_trainable", total - trainable, RUN_ID),
]
spark.createDataFrame(
    dq_rows, "table string, check string, failed_count long, run_id string"
).write.mode("overwrite").parquet(f"{GOLD}/_dq/table=features/ingest_date={window_end}")

print("gold complete", version, "rows", total, "trainable", trainable)
